In [ ]:
dbutils.widgets.text("catalog", "healthcare", "Catalog")
dbutils.widgets.text("bronze_schema", "bronze", "Bronze schema")
dbutils.widgets.text("ml_schema", "ml", "ML schema")
dbutils.widgets.text("etl_service_account", "", "ETL service account")
dbutils.widgets.text("billing_analyst_group", "", "Billing analyst group")

catalog = dbutils.widgets.get("catalog").strip()
bronze_schema = dbutils.widgets.get("bronze_schema").strip()
etl_service_account = dbutils.widgets.get("etl_service_account").strip()
billing_analyst_group = dbutils.widgets.get("billing_analyst_group").strip()
tables = ["claims", "providers", "diagnosis", "cost", "policies"]

In [ ]:
statements = []
for principal in filter(None, [etl_service_account, billing_analyst_group]):
    statements.append(f"GRANT USE CATALOG ON CATALOG `{catalog}` TO `{principal}`")
    statements.append(f"GRANT USE SCHEMA ON SCHEMA `{catalog}`.`{bronze_schema}` TO `{principal}`")

for table in tables:
    fqn = f"`{catalog}`.`{bronze_schema}`.`{table}`"
    if etl_service_account:
        statements.append(f"GRANT INSERT ON TABLE {fqn} TO `{etl_service_account}`")
    if billing_analyst_group:
        statements.append(f"GRANT SELECT ON TABLE {fqn} TO `{billing_analyst_group}`")

for statement in statements:
    spark.sql(statement)
    print(statement)

print(f"OK: grants - statements={len(statements)}")